# Wide-model run: NVIDIA Nemotron ColEmbed 8B (4,096-d) on ViDoRe docvqa

One session = one split: smoke-test the model, encode the 500 pages and their
queries in **float16 over both GPUs**, then run the whole analysis here and export
only a small results zip. The vectors (about 6 GB) never leave Kaggle.

**Before running**
1. *Settings -> Accelerator*: **GPU T4 x2**. *Internet*: **On**.
2. *Save Version -> Save & Run All (Commit)*. Projected 6-9 hours (Kaggle limit: 12).
3. Download `wide_results_docvqa.zip` from the version's *Output* tab.

Differences from the 4B notebook (`notebooks/kaggle_wide_model.ipynb`):
- float16 instead of float32: the 8B is 17.55 GB of weights and only fits two T4s
  in float16. There is **no float32 reference** for it on this hardware, so the
  smoke test cannot compare the two precisions.
- each analysis step has a time budget, so the notebook always reaches the
  packaging cell before Kaggle's 12-hour limit.

Model licence: CC-BY-NC-4.0 (non-commercial research use).

In [ ]:
# parameters
MODEL = "nvidia/nemotron-colembed-vl-8b-v2"
SPLIT = "docvqa"          # this notebook's split
RUN_SMOKE = True
DTYPE = "float16"
DEADLINE_HOURS = 11.25     # everything must be packaged by then (Kaggle stops at 12 h)

In [ ]:
import subprocess, sys, os, json, shutil, glob, time
T0 = time.time()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.device_count() >= 2, "Needs Settings -> Accelerator -> GPU T4 x2 (two GPUs)"
print("torch", torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
total, used, free = shutil.disk_usage("/tmp")
print(f"/tmp free: {free / 1e9:.0f} GB")
assert free > 30e9, "need > 30 GB free in /tmp (18 GB of weights + 6 GB of vectors)"
print(open("/proc/meminfo").read().splitlines()[0])

## 1. Install (the checkout must not be named `optivision`)

In [ ]:
BRANCH = "main"
WORK = "/kaggle/working"
REPO = f"{WORK}/optivision-rag"
DATA = "/tmp/optdata"                      # vectors stay here, outside the output
RESULTS = f"{WORK}/wide_results_{SPLIT}"
os.environ["HF_HOME"] = "/tmp/hf"          # weights next to the vectors, not in the 20 GB output folder
!rm -rf {REPO} && git clone -q --depth 1 --branch {BRANCH} https://github.com/Daemon-VI/optivision-rag.git {REPO}
!cd {REPO} && git log --oneline -1
!pip install -q -e "{REPO}[bench,merge]" 2>&1 | tail -2
# Kaggle preinstalls torchao 0.10; peft refuses to work while an incompatible torchao is importable
!pip uninstall -y -q torchao
!python -c "import torch, transformers, importlib.util as u; print('torch', torch.__version__, '| transformers', transformers.__version__, '| torchao present:', u.find_spec('torchao') is not None)"
os.makedirs(f"{DATA}/vectors", exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

def remaining():
    return DEADLINE_HOURS * 3600 - (time.time() - T0)

def run(cmd, log, env=None, budget=False):
    """Run a step and stop the notebook if it fails. With budget=True the step is
    cut off at the deadline instead, and the notebook goes on to packaging."""
    t = time.time()
    timed_out, code = False, None
    with open(f"{RESULTS}/{log}", "w") as f:
        try:
            code = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True,
                                  env={**os.environ, **(env or {})},
                                  timeout=max(60, remaining()) if budget else None).returncode
        except subprocess.TimeoutExpired:
            timed_out = True
    tail = open(f"{RESULTS}/{log}").read().splitlines()
    print("\n".join(tail[-25:]))
    status = "CUT OFF at the deadline" if timed_out else f"exit {code}"
    print(f"-- {log}: {status} after {(time.time() - t) / 60:.1f} min")
    if timed_out:
        with open(f"{RESULTS}/TIMED_OUT_{log}.txt", "w") as f:
            f.write(f"{log} was cut off after {(time.time() - t) / 3600:.2f} h\n")
        return False
    assert code == 0, f"{log} failed: see the lines above"
    return True

## 2. Smoke test

Loads the model in float16 over both GPUs, encodes 6 pages and their queries, and
checks identity, width, finiteness and norms, own-page retrieval, and our MaxSim
against the model's own `colbert_score`. The encoder refuses non-finite vectors, so
a float16 overflow stops the run here.

In [ ]:
if RUN_SMOKE:
    run([sys.executable, f"{REPO}/scripts/wide_smoke.py", "--model", MODEL, "--multi-gpu", "--dtype", DTYPE,
         "--pages", "6", "--out", f"{RESULTS}/smoke.json"], "smoke.log")
    s = json.load(open(f"{RESULTS}/smoke.json"))
    print({k: s[k] for k in ("dimension", "dtype", "vectors_per_page", "query_retrieves_own_page",
                             "seconds_per_page", "scorer_check", "placement")})
    assert s["dimension"] == 4096, s["dimension"]
    assert s["query_retrieves_own_page"] == s["pages"], "a query did not retrieve its own page"
    print(f"SMOKE OK - estimated encode time for 500 pages: {500 * s['seconds_per_page'] / 60:.0f} min")

## 3. Encode the split (float16 model, both GPUs; vectors are stored as float32)

In [ ]:
DATASET = {"docvqa": "vidore/docvqa_test_subsampled", "infovqa": "vidore/infovqa_test_subsampled"}[SPLIT]
run([sys.executable, f"{REPO}/scripts/encode_vectors.py", MODEL, DATASET, "--backend", "nemotron", "--multi-gpu",
     "--dtype", DTYPE, "--out", f"{DATA}/vectors"], "encode.log")
TAG = f"{MODEL.split('/')[-1]}_{DATASET.split('/')[-1]}"
!ls -la {DATA}/vectors
!rm -rf /tmp/hf    # the weights are no longer needed
print(f"elapsed {(time.time() - T0) / 3600:.2f} h, {remaining() / 3600:.2f} h left")

## 4. Analysis, on the GPU for scoring

`s13_wide_study.py`: baseline, fixed pipelines (codec / token / dimension and their
combinations), memory, latency, two-tier rows, Pareto frontier.
`s7b_selection_rules.py --space=both`: the held-out calibration study on the default
search space and on the projection-extended one. It writes its result only at the
end, so if it is cut off at the deadline this split has the s13 results only.

In [ ]:
ENV = {"OPTIVISION_DATA": DATA, "OPTIVISION_OUT": RESULTS, "OPTIVISION_SCORE_DEVICE": "cuda",
       "OPTIVISION_CACHE_BYTES": str(4 * 1024**3), "PYTHONUNBUFFERED": "1"}
S13_DONE = run([sys.executable, f"{REPO}/scripts/universal_study/s13_wide_study.py", f"vec:{TAG}"],
               "s13_wide.log", ENV, budget=True)
print(f"elapsed {(time.time() - T0) / 3600:.2f} h, {remaining() / 3600:.2f} h left")

In [ ]:
S7B_DONE = False
if S13_DONE and remaining() > 600:
    S7B_DONE = run([sys.executable, f"{REPO}/scripts/universal_study/s7b_selection_rules.py", f"vec:{TAG}",
                    "--space=both"], "s7b_selection.log", ENV, budget=True)
print({"s13_done": S13_DONE, "s7b_done": S7B_DONE, "elapsed_h": round((time.time() - T0) / 3600, 2)})

## 5. Package the results (small)

In [ ]:
shutil.copy(f"{DATA}/vectors/{TAG}_qrels.json", RESULTS)
with open(f"{RESULTS}/run_summary.json", "w") as f:
    json.dump({"model": MODEL, "split": SPLIT, "dtype": DTYPE, "s13_done": S13_DONE, "s7b_done": S7B_DONE,
               "total_hours": round((time.time() - T0) / 3600, 2)}, f, indent=1)
!pip freeze > {RESULTS}/pip_freeze.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > {RESULTS}/gpu.txt
!cd {REPO} && git rev-parse HEAD > {RESULTS}/commit.txt
!rm -rf {REPO}
!find {RESULTS} -type f | sort
!cd {WORK} && rm -f wide_results_{SPLIT}.zip && zip -q -r wide_results_{SPLIT}.zip wide_results_{SPLIT} && ls -la wide_results_{SPLIT}.zip